In [2]:
import pandas as pd
import sqlite3
import glob
import os

In [3]:
df=pd.read_csv(r"C:\Users\Admin\Desktop\PROJECTS\IITG- SQL project\customer_scores_final.csv")

In [4]:
df.head()

,Customer ID,Age,Gender,Location,Category,Frequency of Purchases,Subscription Status,Payment Method,Purchase Amount (USD),Previous Purchases,Review Rating,Promo Code Used,Discount Applied,Season,Age Group,Segment,Value Score
0,1,55,Male,Kentucky,Clothing,Bi-Weekly,Yes,Venmo,53,14,3.1,Yes,Yes,Winter,46-55,Discount Hunter,36.871684
1,2,19,Male,Maine,Clothing,Bi-Weekly,Yes,Cash,64,2,3.1,Yes,Yes,Winter,18-25,Discount Hunter,32.962245
2,3,50,Male,Massachusetts,Clothing,Weekly,Yes,Credit Card,73,23,3.1,Yes,Yes,Spring,46-55,Loyal — Deal User,53.631888
3,4,21,Male,Rhode Island,Footwear,Weekly,Yes,PayPal,90,49,3.5,Yes,Yes,Spring,18-25,Loyal — Deal User,77.262755
4,5,45,Male,Oregon,Clothing,Annually,Yes,PayPal,49,31,2.7,Yes,Yes,Spring,36-45,Loyal — Deal User,28.629847


In [5]:
conn=sqlite3.connect(":memory:")
df.to_sql("customers",conn,index=False, if_exists="replace")
def sql(query):
    return pd.read_sql(query,conn)

In [6]:
q1 = sql("""
SELECT
    CASE
        WHEN "Value Score" >= 60 THEN 'High Value'
        WHEN "Value Score" >= 40 THEN 'Mid Value'
        ELSE 'Low Value'
    END AS Value_Tier,
    COUNT(*) AS customer_count,
    ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2) AS avg_purchases,
    ROUND(AVG("Value Score"), 2) AS avg_score,
    ROUND(100.0 * SUM(CASE WHEN "Promo Code Used" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS promo_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN "Subscription Status" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS subscription_rate_pct
FROM customers
GROUP BY Value_Tier
ORDER BY avg_score DESC
""")
print("Q1 — High vs Low Value Customer Profile")
print(q1.to_string(index=False))

Q1 — High vs Low Value Customer Profile
Value_Tier  customer_count  avg_spend  avg_purchases  avg_score  promo_rate_pct  subscription_rate_pct
High Value             496      80.69          39.99      66.92            36.3                   32.3
 Mid Value            1783      64.29          29.20      48.98            39.9                   29.2
 Low Value            1621      48.39          16.64      29.30            48.4                   22.9


In [7]:
q2 = sql("""
SELECT
    Segment,
    COUNT(*) AS customer_count,
    ROUND(100.0 * SUM(CASE WHEN "Promo Code Used" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS promo_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN "Discount Applied" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS discount_rate_pct,
    ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2) AS avg_purchases,
    ROUND(AVG("Value Score"), 2) AS avg_score
FROM customers
GROUP BY Segment
ORDER BY promo_rate_pct DESC
""")
print("Q2 — Promo Dependency by Segment")
print(q2.to_string(index=False))

Q2 — Promo Dependency by Segment
          Segment  customer_count  promo_rate_pct  discount_rate_pct  avg_spend  avg_purchases  avg_score
Loyal — Deal User            1223           100.0              100.0      59.16          32.55      45.29
  Discount Hunter             454           100.0              100.0      59.59           7.41      29.59
 Loyal — No Deals            1571             0.0                0.0      60.10          32.38      49.08
   Low Engagement             652             0.0                0.0      60.21           7.41      33.87


In [9]:
q3 = sql("""
SELECT
    Location AS state,
    COUNT(*) AS customer_count,
    ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
    ROUND(100.0 * SUM(CASE WHEN "Promo Code Used" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS promo_rate_pct,
    ROUND(AVG("Value Score"), 2) AS avg_value_score,
    CASE
        WHEN AVG("Purchase Amount (USD)") >= 62
         AND 100.0 * SUM(CASE WHEN "Promo Code Used" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) < 40
        THEN 'Organic High Value'
        WHEN AVG("Purchase Amount (USD)") >= 62
         AND 100.0 * SUM(CASE WHEN "Promo Code Used" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) >= 40
        THEN 'High Spend but Promo Driven'
        WHEN AVG("Purchase Amount (USD)") < 62
         AND 100.0 * SUM(CASE WHEN "Promo Code Used" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) < 40
        THEN 'Organic Low Spend'
        ELSE 'Discount Driven'
    END AS geo_profile
FROM customers
GROUP BY Location
HAVING customer_count >= 10
ORDER BY avg_spend DESC

""")
print("Q3 — Geographic Opportunity Map")
print(q3.to_string(index=False))

Q3 — Geographic Opportunity Map
         state  customer_count  avg_spend  promo_rate_pct  avg_value_score                 geo_profile
        Alaska              72      67.60            40.3            48.38 High Spend but Promo Driven
  Pennsylvania              74      66.57            44.6            47.12 High Spend but Promo Driven
       Arizona              65      66.55            33.8            47.31          Organic High Value
 West Virginia              81      63.88            49.4            42.50 High Spend but Promo Driven
        Nevada              87      63.38            47.1            44.70 High Spend but Promo Driven
    Washington              73      63.33            43.8            43.56 High Spend but Promo Driven
  North Dakota              83      62.89            45.8            42.15 High Spend but Promo Driven
      Virginia              77      62.88            37.7            43.91          Organic High Value
          Utah              71      62.58

In [10]:
q4 = sql("""
SELECT
    Category,
    COUNT(*) AS customer_count,
    ROUND(AVG("Previous Purchases"), 2) AS avg_purchase_history,
    ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
    ROUND(100.0 * SUM(CASE WHEN "Previous Purchases" < 15 THEN 1 ELSE 0 END) / COUNT(*), 1) AS new_customer_pct,
    ROUND(100.0 * SUM(CASE WHEN "Previous Purchases" >= 15 THEN 1 ELSE 0 END) / COUNT(*), 1) AS loyal_customer_pct,
    ROUND(100.0 * SUM(CASE WHEN "Promo Code Used" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS promo_rate_pct
FROM customers
GROUP BY Category
ORDER BY avg_purchase_history DESC
""")
print("Q4 — Category: Entry Point vs Retention")
print(q4.to_string(index=False))

Q4 — Category: Entry Point vs Retention
   Category  customer_count  avg_purchase_history  avg_spend  new_customer_pct  loyal_customer_pct  promo_rate_pct
Accessories            1240                 25.73      59.84              27.7                72.3            43.8
   Footwear             599                 25.23      60.26              28.2                71.8            43.2
   Clothing            1737                 25.20      60.03              28.1                71.9            42.1
  Outerwear             324                 24.96      57.17              32.4                67.6            44.4


In [21]:
q5 = sql("""
SELECT
    Gender,
    "Age Group",
    Category,
    "Frequency of Purchases",
    "Payment Method",
    Season,
    COUNT(*) AS customer_count,
    ROUND(AVG("Value Score"), 2) AS avg_value_score,
    ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2) AS avg_purchases,
    ROUND(100.0 * SUM(CASE WHEN "Promo Code Used" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS promo_rate_pct
FROM customers
WHERE Segment = 'Loyal — No Deals'
GROUP BY Gender, "Age Group", Category, "Frequency of Purchases", "Payment Method", Season
ORDER BY avg_value_score DESC
LIMIT 15
""")
print("Q5 — Ideal Customer Profile (Loyal — No Deals segment only)")
print(q5.to_string(index=False))

Q5 — Ideal Customer Profile (Loyal — No Deals segment only)
Gender Age Group    Category Frequency of Purchases Payment Method Season  customer_count  avg_value_score  avg_spend  avg_purchases  promo_rate_pct
Female     26-35    Clothing                 Weekly    Credit Card   Fall               1            87.84       95.0           50.0             0.0
  Male     46-55    Footwear                 Weekly    Credit Card Summer               1            85.79      100.0           49.0             0.0
  Male     46-55    Clothing              Bi-Weekly     Debit Card Winter               1            81.64       95.0           50.0             0.0
Female     36-45 Accessories                 Weekly          Venmo Winter               1            81.59       75.0           50.0             0.0
Female     46-55    Footwear                 Weekly  Bank Transfer   Fall               1            80.45       87.0           43.0             0.0
  Male       55+    Footwear              Bi-W